# 06 · Reproducción del campeón (notebook limpio) y exportación de corridas

Este notebook no entrena nada: carga `models:/anticipar-mora-modelo@champion` desde el registro, reconstruye el mismo hold-out (semilla 42, mismo snapshot verificado por hash) y comprueba que la AUC de prueba sea **exactamente** la registrada. Al final exporta todas las corridas a `docs/corridas.csv`.

In [1]:
import sys; sys.path.insert(0, '..')
import mlflow, pandas as pd
from mlflow import MlflowClient
from sklearn.metrics import roc_auc_score
from src.datos import cargar_obt, dividir
from src import tracking
tracking.configurar(); cliente = MlflowClient()
version = cliente.get_model_version_by_alias(tracking.MODELO_REGISTRO, 'champion')
run = cliente.get_run(version.run_id)
print(f"Campeón: {tracking.MODELO_REGISTRO} v{version.version} | run {version.run_id} | modelo {run.data.tags['modelo']}")

Campeón: anticipar-mora-modelo v1 | run 3933132ad8a945c4bf1249f8dee5b655 | modelo lightgbm


In [2]:
obt, meta = cargar_obt(); X_train, X_test, y_train, y_test = dividir(obt)
assert meta['hash_dataset'] == run.data.params['datos_hash'], 'El snapshot cambió: no es el mismo dataset'
modelo = mlflow.sklearn.load_model(f'models:/{tracking.MODELO_REGISTRO}@champion')
auc = roc_auc_score(y_test, modelo.predict_proba(X_test)[:, 1])
registrada = run.data.metrics['test_roc_auc']
print(f'AUC de prueba reproducida: {auc:.10f}\nAUC de prueba registrada:  {registrada:.10f}\nDiferencia: {abs(auc-registrada):.2e}')
assert abs(auc - registrada) < 1e-9
print('OK: el modelo del registro reproduce exactamente la métrica registrada')

AUC de prueba reproducida: 0.7828591222
AUC de prueba registrada:  0.7828591222
Diferencia: 0.00e+00
OK: el modelo del registro reproduce exactamente la métrica registrada


In [3]:
pyf = mlflow.pyfunc.load_model(f'models:/{tracking.MODELO_REGISTRO}@champion')   # misma predicción por la interfaz genérica (PSet 4)
import numpy as np
print('pyfunc AUC:', round(roc_auc_score(y_test, np.asarray(pyf.predict(X_test))[:, 1]), 6))

pyfunc AUC: 0.782859


## Exportación de todas las corridas

In [4]:
exp = mlflow.get_experiment_by_name(tracking.EXPERIMENTO)
corridas = mlflow.search_runs([exp.experiment_id], max_results=5000)
corridas.to_csv('../docs/corridas.csv', index=False)
print(len(corridas), 'corridas exportadas a docs/corridas.csv')
corridas['tags.fase'].value_counts()

104 corridas exportadas a docs/corridas.csv


tags.fase
tuning_config    90
candidato         7
tuning            3
final             2
baseline          2
Name: count, dtype: int64